# TD 1 — Using models through their API 🔌

In this lab, we will connect from Python to a **hosted** LLM: the model runs on a provider's servers; Python sends it a request over the internet and receives a response.

We'll use Google AI Studio as an example, because its free tier allows for easy experimentation. The same principles apply to other providers.

**At the end of this lab, you will know how to:**

- call an LLM with an SDK, then directly over HTTP;
- hold a conversation with a model that has no memory;
- estimate a cost;
- get a structured answer and validate it;
- handle API errors;
- write a precise prompt.

## 1. What happens when we call an LLM API?

The request contains:

- an **API key**, which identifies you;
- a **model** identifier;
- some **content**: a simple string, or the list of messages of a conversation.

The response contains the generated text and metadata, such as the number of tokens used.

The official `google-genai` Python package is an **SDK**: it provides objects and methods that build these requests for us. In part 5, we will send an HTTP request ourselves to see what it does.

## 2. Create an account and an API key 🔐

1. Go to [Google AI Studio](https://aistudio.google.com/apikey). Log in or create an account.
2. Create a new key and copy it straight away.
3. Paste the key into a new file called `.env`, in the current folder (the leading dot makes the file hidden on Linux and macOS). The file must contain a single line:

```
GEMINI_API_KEY="YOUR_KEY_HERE"
```

4. If you use Git, add `.env` to your `.gitignore` file.

> **WARNING**: a key is a **secret credential**. Never paste it into a cell, commit it, or show it in a screenshot. If it leaks, revoke it and create another one.

### What about your data?

According to [Google's terms](https://aistudio.google.com/docs/terms) (read in November 2026), content sent to the free tier may be used to improve its products and may be read by people.
In any case: **never send personal or confidential data without authorisation.**

## 3. Install the packages 📦

- `google-genai`: call the API (beware, `google-generative-ai` is the old SDK);
- `python-dotenv`: read the `.env` file;
- `requests`: send an HTTP request.

**With pip**: the next cell is in "raw" format. Change it to "code" to run it, only if the packages are not installed yet.

**With uv**: open a terminal in the project folder and run `uv add google-genai python-dotenv requests`.

### Make the key available to Python 🔐

`load_dotenv()` reads the `.env` file and puts the key in the environment variables, so it never appears in the code.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY")

# We check that the key exists, but we never print it!
if api_key:
    print("Key found.")
else:
    print("No key found: check your .env file, then restart the kernel.")

## 4. A first call 🤖

The list of models is [here](https://aistudio.google.com/docs/models). It changes often: if the model below no longer exists, replace it.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)
MODEL = "gemini-3.5-flash-lite"

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Why is the sky blue? Answer in two sentences.",
)
print(response.text)

The `response` object contains more than the text. Let's display all of it:

In [ ]:
response

### Demonstration: read the metadata 🔎

From the `response` object, let's display the model version, the reason why it stopped (`finish_reason`) and the number of tokens in the question, in the answer, and the total.

In [ ]:
usage = response.usage_metadata

print("Model         :", response.model_version)
print("Finish reason :", response.candidates[0].finish_reason)
print("Input tokens  :", usage.prompt_token_count)
print("Output tokens :", usage.candidates_token_count)
print("Total tokens  :", usage.total_token_count)

## 5. Under the hood: the HTTP request 🌐

There is nothing magical about the SDK: it sends a `POST` request containing JSON. Let's do the same with `requests`.

Note that the key travels in a **header**, not in the address: addresses are often recorded in logs.

In [ ]:
import requests

API_URL = "https://generativelanguage.googleapis.com/v1beta"

payload = {
    "contents": [
        {"role": "user", "parts": [{"text": "Why is the sky blue? Answer in two sentences."}]}
    ]
}

http_response = requests.post(
    f"{API_URL}/models/{MODEL}:generateContent",
    headers={"x-goog-api-key": api_key},
    json=payload,
    timeout=60,
)

print(http_response.status_code)
http_response.json()

### Demonstration: our own small SDK 🔧

The `ask_http(prompt)` function sends a prompt and returns **only the text** of the answer. If there is an HTTP error, it raises an exception. The text may be split into several `parts`: it joins them.

In [ ]:
def ask_http(prompt):
    payload = {"contents": [{"role": "user", "parts": [{"text": prompt}]}]}
    r = requests.post(
        f"{API_URL}/models/{MODEL}:generateContent",
        headers={"x-goog-api-key": api_key},
        json=payload,
        timeout=60,
    )
    r.raise_for_status()
    parts = r.json()["candidates"][0]["content"]["parts"]
    return "".join(part.get("text", "") for part in parts)

print(ask_http("Say hello in three languages."))

## 6. A conversation: the model has no memory 🧠

Between two requests, the model remembers **nothing**. To hold a conversation, you must send all the previous messages again. Each message has a role: `user` (you) or `model` (the model).

In [ ]:
conversation = [
    {"role": "user", "parts": [{"text": "I am going to Marseille for three days."}]},
    {"role": "model", "parts": [{"text": "Great! Do you like hiking?"}]},
    {"role": "user", "parts": [{"text": "Yes. How many days do I have, and where am I going?"}]},
]

response = client.models.generate_content(model=MODEL, contents=conversation)
print(response.text)

### Demonstration: with and without history 🧾

1. We send "My name is Ada.", then, in a **second, independent request**, "What is my name?". The model cannot know the answer.
2. The `chat(history, message)` function adds the message to the history, calls the model, then adds its answer to the history. This time, the model finds the name.

In [ ]:
# 1. Two independent requests: the model cannot know the name
client.models.generate_content(model=MODEL, contents="My name is Ada.")
print(client.models.generate_content(model=MODEL, contents="What is my name?").text)

# 2. With a history
def chat(history, message):
    history.append({"role": "user", "parts": [{"text": message}]})
    answer = client.models.generate_content(model=MODEL, contents=history).text
    history.append({"role": "model", "parts": [{"text": answer}]})
    return answer

history = []
print(chat(history, "My name is Ada."))
print(chat(history, "What is my name?"))
print(len(history), "messages in the history")

## 7. How much does it cost? 💰

You pay per **token**, and generated tokens cost more than tokens read. [Price](https://aistudio.google.com/docs/pricing) of `gemini-3.5-flash-lite`, read in November 2026:

| | Free tier | Paid tier (per million tokens) |
| --- | --- | --- |
| Input | free | $0.30 |
| Output (thinking included) | free | $2.50 |

The free tier is limited. [Limits](https://aistudio.google.com/docs/rate-limits) are measured in requests per minute (RPM), input tokens per minute (TPM) and requests per day (RPD). They apply **per project**, not per key. Beyond them, the API returns a 429 error.

### Demonstration: compute a cost 🧮

We want to summarise 10,000 texts with the paid tier. Each request reads 500 tokens and generates 200. How much would it cost, and what share of the price comes from the generated tokens?

In [ ]:
PRICE_IN, PRICE_OUT = 0.30, 2.50  # dollars per million tokens

def cost(input_tokens, output_tokens):
    return (input_tokens * PRICE_IN + output_tokens * PRICE_OUT) / 1_000_000

total = cost(10_000 * 500, 10_000 * 200)
output_only = cost(0, 10_000 * 200)
print(f"Total: ${total:.2f}, including {output_only / total:.0%} for the output")

## 8. Parameters ⚙️

**Role**

- `system_instruction`: gives the model a role, a tone or rules.

**Choosing the next token**

- `temperature`: the lower it is, the more the model picks the most probable tokens. Even at `0`, an identical result is not guaranteed.
- `top_p`: only keeps the smallest set of tokens whose cumulative probability reaches `top_p`.
- `top_k`: only keeps the `top_k` most probable tokens.
- `seed`: fixes the randomness, to make answers more reproducible (without guarantee).

> **Watch out, this is changing!** According to [Google](https://aistudio.google.com/docs/text-generation?generate-content=true), Gemini 3.6 and later models **ignore** `temperature`, `top_p` and `top_k`. For earlier Gemini 3 models, the advice is to keep the default values. We change them here only to understand. TD 2 runs the experiment again on a local model, where these settings remain entirely in our hands.

**Length**

- `max_output_tokens`: maximum number of generated tokens. The text may be cut off abruptly.
- `stop_sequences`: strings that stop the generation.

**Format, reasoning, tools**

- `response_mime_type` and `response_schema`: force a JSON output (part 9).
- `thinking_config`: sets the reasoning effort.
- `tools` and `tool_config`: give tools to the model (TD 2 and TD 6).
- `safety_settings`: sets the content filters.

### Experiment: temperature 🌡️

A good experiment changes **only one thing at a time**. Here, only the temperature varies: we do not set a seed.

In [ ]:
from time import sleep

QUESTION = "Invent the name of a bakery in Marseille. Answer with the name only."

def generate(prompt, **parameters):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(**parameters),
    )
    return response.text

for temperature in (0, 1, 2):
    answers = []
    for _ in range(4):
        answers.append(generate(QUESTION, temperature=temperature).strip())
        sleep(1)  # avoid the rate limit
    print(f"temperature={temperature}: {len(set(answers))} different answers out of 4 -> {answers}")

### Demonstration: the seed 🌱

Let's keep the default temperature and ask the question four times, first without a seed, then with `seed=42`. Each time, we count the different answers.

In [ ]:
for parameters in ({}, {"seed": 42}):
    answers = []
    for _ in range(4):
        answers.append(generate(QUESTION, **parameters).strip())
        sleep(1)
    print(parameters, len(set(answers)), "different answers ->", answers)

The seed often makes the answers identical, but this is **not guaranteed**: the provider only promises a "best effort". The hardware or the model version may change the result.

## 9. Structured output 🧩

Free text is hard to use in a program. We can force a **schema** with Pydantic, then convert the answer into a Python object.

Always check `finish_reason`: if the answer was cut off (`MAX_TOKENS`), the JSON is incomplete and therefore invalid. A **valid** JSON may also contain **false** information.

In [ ]:
from pydantic import BaseModel

class Summary(BaseModel):
    overview: str
    key_points: list[str]

response = client.models.generate_content(
    model=MODEL,
    contents="Explain in a few words why Marseille is worth visiting.",
    config=types.GenerateContentConfig(
        system_instruction="You are a travel expert. Be concise.",
        response_mime_type="application/json",
        response_schema=Summary,
    ),
)

print(response.candidates[0].finish_reason)

summary = Summary.model_validate_json(response.text)  # raises an error if the JSON is invalid
print(summary.overview)
for point in summary.key_points:
    print("-", point)

### Demonstration: extract information 🎒

The `Trip` schema describes a trip: city, number of days, means of transport and budget. The budget is not in the text: it must be `None`, because the model must not invent it.

In [ ]:
class Trip(BaseModel):
    city: str | None
    days: int | None
    transport: str | None
    budget: float | None

text = "We are going to Nice for four days, by train, in early June."

response = client.models.generate_content(
    model=MODEL,
    contents=f"Extract the trip information. Use null when it is not in the text.\n\n{text}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Trip,
    ),
)
trip = Trip.model_validate_json(response.text)
print(trip)

## 10. Handling errors 🚧

A remote API can fail. The most common errors:

| Code | Meaning | What to do? |
| --- | --- | --- |
| 400 | malformed request | fix the code |
| 401 / 403 | invalid key, or no permission | check the key |
| 429 | too many requests | wait, then try again |
| 500 / 503 | server down or overloaded | wait, then try again |

The good practice is to wait longer and longer between attempts: 1 s, 2 s, 4 s... This is called **exponential backoff**.

### Demonstration: try again at the right time 🔁

The `ask_with_retry(prompt, max_tries=4)` function tries again **only** for codes 429, 500 and 503, with exponential backoff. Other errors are raised immediately: trying again would be useless. In the SDK, errors are of type `errors.APIError` and have a `code` attribute.

In [ ]:
from google.genai import errors

def ask_with_retry(prompt, max_tries=4):
    for attempt in range(max_tries):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt).text
        except errors.APIError as error:
            last_try = attempt == max_tries - 1
            if error.code not in (429, 500, 503) or last_try:
                raise
            wait = 2 ** attempt
            print(f"Error {error.code}, new attempt in {wait} s")
            sleep(wait)

print(ask_with_retry("Say hello."))

## 11. Writing a good prompt ✍️

A precise prompt usually contains:

1. the **context**: who is speaking, to whom, for what purpose;
2. the **task**, with definitions;
3. **examples** (*few-shot*) if the task is ambiguous;
4. the expected **format**;
5. the **right to answer "I don't know"**.

Let's compare a vague prompt with a precise one:

In [ ]:
sentence = "On voyait le cap Canaille au loin ; cette fois, nous avons renoncé à y aller."

vague = f"What about the place in this sentence? {sentence}"

precise = f"""You help a geographer who studies travel stories.
Did the traveller go to the place named in the sentence?
Answer with one word only: visited, seen, or unknown.
- visited: the traveller went there
- seen: the traveller only saw it from elsewhere
- unknown: the sentence does not say

Sentence: {sentence}"""

for prompt in (vague, precise):
    print(generate(prompt))
    print("---")

### Demonstration: with and without examples 🎯

Here are five sentences and the expected answer. We count the right answers with a prompt **without any example** (*zero-shot*), then with two **examples** (*few-shot*), different from the five sentences.

In [ ]:
sentences = [
    ("Je suis montée à la citadelle de Sisteron.", "visited"),
    ("À La Grave, j'ai regardé la Meije depuis le village.", "seen"),
    ("J'aurais aimé monter dans la vallée des Merveilles, mais je n'avais pas le temps.", "considered"),
    ("On voyait le cap Canaille au loin.", "seen"),
    ("J'ai traversé La Seyne-sur-Mer.", "visited"),
]

In [ ]:
rules = """Did the traveller go to the place? Answer with one word only:
visited (went there), seen (only saw it), considered (thought about it but did not go).
"""
examples = """
Sentence: Nous avons dîné à Cassis. -> visited
Sentence: Depuis le bateau, on apercevait Porquerolles. -> seen
"""

def score(prompt_start):
    correct = 0
    for sentence, expected in sentences:
        answer = generate(f"{prompt_start}\nSentence: {sentence} ->").strip().lower()
        correct += answer == expected
        sleep(1)
    return correct

print("zero-shot:", score(rules), "/ 5")
print("few-shot :", score(rules + examples), "/ 5")

The examples must not be part of the tested sentences: the model would only have to copy the answer, and the score would no longer say anything about new sentences.

> Five sentences are not enough to draw a conclusion. TD 4 will show how to **evaluate** a model seriously.

## Cheat sheet 📋

| Goal | Parameter(s) |
| --- | --- |
| Set the role and the rules | `system_instruction` |
| Force a valid JSON | `response_mime_type` + `response_schema` |
| Limit length and cost | `max_output_tokens`, `stop_sequences` |
| Make answers more stable | `seed` (and `temperature` on the models that accept it) |
| Connect functions or data | `tools`, `tool_config` |

**To make a result reproducible, always write down:** the name and version of the model, the date, the exact prompt and the parameters.